# Discovery Pipeline Debug
Inspect dedup, relevance filtering, clustering, and ranking quality.

In [ ]:
import json, sys, os
import numpy as np
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
sys.path.insert(0, '.')

from embedding import embed_texts, cosine_similarity_matrix, _item_text, DEDUP_NEEDS_REVIEW
from models import RawItem
from datetime import datetime

# Load raw items from last run
raw = json.loads(open('../data/temp/run_2026-03-01_0118/raw_items.json').read())
items = [
    RawItem(
        title=r['title'], url=r['url'], source_type=r['source_type'],
        source_id=r['source_id'], timestamp=datetime.fromisoformat(r['timestamp']),
        metadata=r['metadata'], content_snippet=r['content_snippet'],
        raw_content=r['raw_content']
    ) for r in raw
]
print(f'Loaded {len(items)} items')

## 1. Embed all items

In [ ]:
texts = [_item_text(item) for item in items]
vectors = embed_texts(texts)
V = np.array(vectors)
print(f'Embedded {len(vectors)} items, dim={V.shape[1]}')

## 2. Dedup — all pairs above threshold
Shows every pair with cosine >= 0.80 (the merge threshold). These are the items the pipeline considers duplicates.

In [ ]:
sim = cosine_similarity_matrix(V)

# Find all pairs above threshold
pairs = []
for i in range(len(items)):
    for j in range(i+1, len(items)):
        if sim[i,j] >= DEDUP_NEEDS_REVIEW:
            pairs.append((i, j, sim[i,j]))

pairs.sort(key=lambda x: -x[2])
print(f'Found {len(pairs)} duplicate pairs (cosine >= {DEDUP_NEEDS_REVIEW})\n')

for i, j, s in pairs:
    a, b = items[i], items[j]
    print(f'--- cosine: {s:.3f} ---')
    print(f'  A [{i}] ({a.source_type}/{a.source_id}) {a.title[:80]}')
    print(f'  B [{j}] ({b.source_type}/{b.source_id}) {b.title[:80]}')
    print(f'  URLs match: {a.url == b.url}')
    print()

## 3. Near-misses (0.65 - 0.80)
Items that are similar but NOT merged. Check if any should have been.

In [ ]:
near = []
for i in range(len(items)):
    for j in range(i+1, len(items)):
        if 0.65 <= sim[i,j] < DEDUP_NEEDS_REVIEW:
            near.append((i, j, sim[i,j]))

near.sort(key=lambda x: -x[2])
print(f'Found {len(near)} near-miss pairs (0.65 <= cosine < {DEDUP_NEEDS_REVIEW})\n')

for i, j, s in near:
    a, b = items[i], items[j]
    print(f'--- cosine: {s:.3f} ---')
    print(f'  A [{i}] ({a.source_type}/{a.source_id}) {a.title[:80]}')
    print(f'  B [{j}] ({b.source_type}/{b.source_id}) {b.title[:80]}')
    print()

## 4. Relevance scores against profile
Cosine similarity of each item against the user's interest vector.

In [ ]:
from embedding import cosine_similarity, RELEVANCE_THRESHOLD
import yaml

profile = yaml.safe_load(open('../data/profiles/default/profile.yaml'))
field = profile.get('field', '')
focus = profile.get('focus_areas', [])
interest_text = f"{field} {' '.join(focus)}"
interest_vec = np.array(embed_texts([interest_text])[0])

print(f'Interest text: "{interest_text}"')
print(f'Threshold: {RELEVANCE_THRESHOLD}\n')

scored = []
for i, item in enumerate(items):
    s = cosine_similarity(V[i], interest_vec)
    scored.append((i, s, item))

scored.sort(key=lambda x: -x[1])

print('=== MOST RELEVANT ===')
for i, s, item in scored[:10]:
    mark = 'PASS' if s >= RELEVANCE_THRESHOLD else 'DROP'
    print(f'  [{mark}] {s:.3f}  [{i}] ({item.source_type}) {item.title[:70]}')

dropped = [(i,s,item) for i,s,item in scored if s < RELEVANCE_THRESHOLD]
if dropped:
    print(f'\n=== DROPPED ({len(dropped)} items below {RELEVANCE_THRESHOLD}) ===')
    for i, s, item in dropped:
        print(f'  {s:.3f}  [{i}] ({item.source_type}) {item.title[:70]}')
else:
    print(f'\nNo items dropped (all above {RELEVANCE_THRESHOLD})')

## 5. Cluster contents
What items ended up in each cluster?

In [ ]:
from embedding import dedup_items, embed_items, filter_relevance, cluster_items

# Re-run pipeline steps to get clusters
items_with_emb = list(zip(items, vectors))
deduped = dedup_items(items_with_emb)
print(f'After dedup: {len(deduped)} (removed {len(items) - len(deduped)})')

emb_lookup = {id(item): vec for item, vec in items_with_emb}
deduped_with_emb = [(item, emb_lookup[id(item)]) for item in deduped]

relevant = filter_relevance(deduped_with_emb, profile)
print(f'After relevance: {len(relevant)}')

relevant_with_emb = [(item, emb_lookup[id(item)]) for item in relevant]
clusters = cluster_items(relevant_with_emb, n_clusters=8)
print(f'Clusters: {len(clusters)}\n')

for c in clusters:
    print(f'=== Cluster {c.cluster_id}: "{c.representative_title[:60]}" ({len(c.items)} items) ===')
    for item in c.items:
        print(f'  - ({item.source_type}/{item.source_id}) {item.title[:70]}')
    print()